<a href="https://colab.research.google.com/github/pnperl/Equity-Intelligence/blob/main/Equity_Intelligence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Here is the `unified_pipeline.py` file, synthesized from your codebase. I have resolved duplicate technical analysis logic, merged ML and News functions, and structured everything cleanly using Object-Oriented Programming (OOP) to prevent namespace collisions.

In [ ]:
!pip install mplfinance
!pip install ta-lib
!pip install ta
!pip install feedparser
!pip install vaderSentiment

import os
import re
import time
import json
import logging
import warnings
import urllib.parse
import html
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import mplfinance as mpf
from tqdm import tqdm
from scipy.signal import argrelextrema

import talib
from ta.momentum import RSIIndicator, StochasticOscillator, ROCIndicator
from ta.trend import MACD
from ta.volatility import BollingerBands, AverageTrueRange
from ta.volume import VolumeWeightedAveragePrice, OnBalanceVolumeIndicator

import feedparser
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score
import joblib

import torch
import torch.nn as nn

# Suppress environment warnings
warnings.filterwarnings('ignore', category=FutureWarning)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

# ==========================================
# 1. GLOBAL CONFIGURATIONS
# ==========================================
CONFIG = {
    'timezone': 'Asia/Kolkata',
    'default_period': '1y',
    'default_days': 730,
    'tickers': ['RELIANCE.NS', 'TCS.NS', 'HDFCBANK.NS'],
    'model_path': './models/',
    'plot_path': './plots/',
    'finnhub_api_key': 'YOUR_FINNHUB_API_KEY',
    'openai_api_key': 'YOUR_OPENAI_API_KEY',
    'swing_distance': 6,
    'vol_spike_factor': 1.5,
    'min_history_days': 60
}

# Ensure directories exist
os.makedirs(CONFIG['model_path'], exist_ok=True)
os.makedirs(CONFIG['plot_path'], exist_ok=True)


# ==========================================
# 2. DATA PROCESSOR
# ==========================================
class DataProcessor:
    """Handles stock data retrieval and technical indicator calculations."""

    @staticmethod
    def fetch_data(ticker, period="1y"):
        try:
            df = yf.download(ticker, period=period, auto_adjust=True, progress=False)
            if df.empty:
                return pd.DataFrame()
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = [col[0] for col in df.columns]
            df.dropna(subset=['Close'], inplace=True)
            return df
        except Exception as e:
            logger.error(f"Error fetching data for {ticker}: {e}")
            return pd.DataFrame()

    @staticmethod
    def compute_indicators(df):
        if df.empty or len(df) < CONFIG['min_history_days']:
            return pd.DataFrame()

        df = df.copy()
        # Moving Averages
        df['MA20'] = df['Close'].rolling(window=20).mean()
        df['MA50'] = df['Close'].rolling(window=50).mean()
        df['MA200'] = df['Close'].rolling(window=200).mean()

        # Core Technicals via TA-Lib
        try:
            df['EMA12'] = talib.EMA(df['Close'], timeperiod=12)
            df['EMA26'] = talib.EMA(df['Close'], timeperiod=26)
            df['MACD'], df['MACD_Signal'], df['MACD_Hist'] = talib.MACD(df['High'], df['Low'], df['Close'], fastperiod=12, slowperiod=26, signalperiod=9)
            df['RSI'] = talib.RSI(df['Close'], timeperiod=14)
            df['Stochastic_K'], df['Stochastic_D'] = talib.STOCH(df['High'], df['Low'], df['Close'])
            df['ADX'] = talib.ADX(df['High'], df['Low'], df['Close'], timeperiod=14)
            df['ATR'] = talib.ATR(df['High'], df['Low'], df['Close'], timeperiod=14)
            df['OBV'] = talib.OBV(df['Close'], df['Volume'])
        except Exception as e:
            logger.warning(f"TA-Lib failed, falling back to manual ta library: {e}")
            df['RSI'] = RSIIndicator(close=df['Close']).rsi()
            macd = MACD(close=df['Close'])
            df['MACD'], df['MACD_Signal'] = macd.macd(), macd.macd_signal()

        # Bollinger Bands
        df['BB_Mid'] = df['Close'].rolling(window=20).mean()
        df['BB_Std'] = df['Close'].rolling(window=20).std()
        df['BB_Upper'] = df['BB_Mid'] + (2 * df['BB_Std'])
        df['BB_Lower'] = df['BB_Mid'] - (2 * df['BB_Std'])

        # Volatility / Breakout Detection
        df['Vol20'] = df['Volume'].rolling(20).mean()
        df['Is_Volume_Breakout'] = df['Volume'] > (CONFIG['vol_spike_factor'] * df['Vol20'])

        return df.dropna()

    @staticmethod
    def detect_support_resistance(df, period=60):
        recent = df.tail(period).copy()
        if recent.empty or len(recent) < 5:
            return [], []
        support = recent['Low'].nsmallest(2).values.tolist()
        resistance = recent['High'].nlargest(2).values.tolist()
        return list(set(support)), list(set(resistance))


# ==========================================
# 3. NEWS & SENTIMENT ANALYZER
# ==========================================
class NewsAnalyzer:
    """Fetches Google/Yahoo RSS feeds and performs sentiment analysis."""

    def __init__(self):
        self.analyzer = SentimentIntensityAnalyzer()

    def get_stock_news(self, ticker, max_items=5):
        company_name = self._get_company_name(ticker)
        company_name_q = urllib.parse.quote(company_name)
        ticker_q = urllib.parse.quote(ticker)

        sources = [
            f"https://feeds.finance.yahoo.com/rss/2.0/headline?s={ticker_q}&region=US&lang=en-US",
            f"https://news.google.com/rss/search?q={company_name_q}"
        ]

        news_items = []
        for url in sources:
            d = feedparser.parse(url)
            if d.entries:
                for entry in d.entries[:max_items]:
                    title = html.escape(entry.get('title', ''))
                    desc = html.escape(entry.get('summary', entry.get('description', '')))
                    score = self.analyzer.polarity_scores(title + " " + desc)['compound']
                    news_items.append({
                        'title': title,
                        'link': entry.get('link', ''),
                        'published': entry.get('published', 'N/A'),
                        'sentiment': score
                    })
                break  # Stop if primary feed works

        avg_sentiment = np.mean([n['sentiment'] for n in news_items]) if news_items else 0
        return news_items, avg_sentiment

    def _get_company_name(self, ticker):
        try:
            return yf.Ticker(ticker).info.get('longName', ticker)
        except:
            return ticker


# ==========================================
# 4. MODEL TRAINER (ML & Backtesting)
# ==========================================
class LSTMNetwork(nn.Module):
    def __init__(self, input_size=3, hidden_size=50):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        _, (h, _) = self.lstm(x)
        return self.fc(h[-1])

class ModelTrainer:
    """Handles RandomForest training and LSTM execution."""

    @staticmethod
    def train_rf_classifier(df, ticker):
        features = ['RSI', 'MACD', 'BB_Upper', 'BB_Lower', 'Close']
        df = df.copy()

        # Target: Is price higher 5 days from now?
        df['target'] = np.sign(df['Close'].shift(-5) - df['Close'])
        df.dropna(inplace=True)

        X = df[features]
        y = df['target']
        if len(X) < 30 or len(set(y)) < 2:
            return None, 0.0

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        clf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
        clf.fit(X_train, y_train)
        acc = accuracy_score(y_test, clf.predict(X_test))

        # Export Model
        model_path = os.path.join(CONFIG['model_path'], f"{ticker}_rf.pkl")
        joblib.dump(clf, model_path)
        return clf, acc


# ==========================================
# 5. REPORT GENERATOR
# ==========================================
class ReportGenerator:
    """Manages Chart Generation and CLI/HTML Output Summaries."""

    @staticmethod
    def plot_chart(ticker, df, support, resistance):
        if len(df) < 20:
            return None

        plot_df = df.tail(120) # Look at last 120 days
        apds = []

        # Append moving averages
        apds.append(mpf.make_addplot(plot_df['MA20'], color='blue', width=0.7))
        apds.append(mpf.make_addplot(plot_df['MA50'], color='orange', width=0.7))

        # Append Support / Resistance lines
        for s in support:
            apds.append(mpf.make_addplot([s]*len(plot_df), color='green', linestyle='--', width=0.5))
        for r in resistance:
            apds.append(mpf.make_addplot([r]*len(plot_df), color='red', linestyle='--', width=0.5))

        figpath = os.path.join(CONFIG['plot_path'], f"{ticker}_chart.png")
        mpf.plot(plot_df, type='candle', style='yahoo', volume=True, addplot=apds,
                 title=f"{ticker} Technical Analysis", savefig=figpath)
        return figpath

    @staticmethod
    def print_cli_summary(ticker, df, news_items, avg_sentiment, acc):
        last = df.iloc[-1]

        # Simple Bullish Logic
        bullish = (last['Close'] > last['MA50']) and (last['RSI'] > 50) and (last['MACD'] > last['MACD_Signal'])
        outlook = "Bullish" if bullish else "Bearish / Neutral"

        print(f"\n{'='*60}")
        print(f"📊 REPORT: {ticker}")
        print(f"{'='*60}")
        print(f"Current Close  : {last['Close']:.2f}")
        print(f"AI/Tech Outlook: {outlook}")
        print(f"RSI            : {last['RSI']:.2f} | MACD: {last['MACD']:.2f}")
        print(f"Vol Breakout   : {'Yes' if last['Is_Volume_Breakout'] else 'No'}")
        print(f"News Sentiment : {avg_sentiment:.2f} (Scale -1 to 1)")
        print(f"Model Accuracy : {acc:.2%}")
        print("\n📰 Recent News:")
        for n in news_items[:3]:
            print(f"  - {n['title']} (Sent: {n['sentiment']:.2f})")
        print(f"{'='*60}\n")


# ==========================================
# 6. UNIFIED PIPELINE (Entry Point)
# ==========================================
def main():
    logger.info("Starting Unified Stock Analysis Pipeline...")

    # Initialize Classes
    dp = DataProcessor()
    na = NewsAnalyzer()
    mt = ModelTrainer()
    rg = ReportGenerator()

    for ticker in CONFIG['tickers']:
        logger.info(f"Analyzing {ticker}...")

        # 1. Fetch & Process Data
        df = dp.fetch_data(ticker, period=CONFIG['default_period'])
        if df.empty:
            logger.warning(f"No valid data retrieved for {ticker}. Skipping.")
            continue

        df = dp.compute_indicators(df)
        support, resistance = dp.detect_support_resistance(df)

        # 2. Machine Learning & Sentiment Integration
        news_items, avg_sent = na.get_stock_news(ticker)
        model, acc = mt.train_rf_classifier(df, ticker)

        # 3. Output & Export
        figpath = rg.plot_chart(ticker, df, support, resistance)
        rg.print_cli_summary(ticker, df, news_items, avg_sent, acc)

    logger.info("Pipeline execution complete.")

if __name__ == "__main__":
    main()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 3.9 MB/s eta 0:00:00



📊 REPORT: RELIANCE.NS
Current Close  : 1320.00
AI/Tech Outlook: Bearish / Neutral
RSI            : 38.18 | MACD: -13.54
Vol Breakout   : No
News Sentiment : 0.57 (Scale -1 to 1)
Model Accuracy : 66.67%

📰 Recent News:
  - A historic milestone for Indian industry. Reliance Industries has become the first Indian company to cross US$10 billion in annual net profit — ₹95,754 crore in FY 2025-26, up 17.8% Y-o-Y. Consolidated revenues stood at ₹11,75,919 crore and EBITDA at ₹2, - Facebook (Sent: 0.70)
  - India&#x27;s Reliance Industries paid Trump $10 million before he took office. It keeps getting wins from Trump. - CREW - Citizens for Responsibility and Ethics in Washington (Sent: 0.81)
  - Is It Smart To Buy Reliance Industries Limited (NSE:RELIANCE) Before It Goes Ex-Dividend? - simplywall.st (Sent: 0.48)




📊 REPORT: TCS.NS
Current Close  : 2297.40
AI/Tech Outlook: Bearish / Neutral
RSI            : 44.51 | MACD: -37.03
Vol Breakout   : Yes
News Sentiment : 0.58 (Scale -1 to 1)
Model Accuracy : 88.89%

📰 Recent News:
  - TCS and Mistral agree partnership for advanced AI model deployment (Sent: 0.74)
  - Is Amazon.com (AMZN) the Best Strong Buy Stock to Buy and Hold for the Next 5 Years? (Sent: 0.93)
  - The Philippines Enterprise ICT Intelligence Report 2025 Featuring Microsoft, Globe Telecom, IBM, Converge ICT Solutions, and Tata Consultancy Services (TCS) (Sent: 0.95)




📊 REPORT: HDFCBANK.NS
Current Close  : 742.70
AI/Tech Outlook: Bearish / Neutral
RSI            : 38.19 | MACD: -9.16
Vol Breakout   : No
News Sentiment : 0.04 (Scale -1 to 1)
Model Accuracy : 55.56%

📰 Recent News:
  - India&#x27;s HDFC Bank falls on report of payments to attract big deposits - Reuters (Sent: 0.61)
  - Investors Fear More Skeletons May Emerge at HDFC Bank - Bloomberg.com (Sent: -0.75)
  - HDFC Bank (NYSE: HDB) addresses media report on ₹45 crore audit matter - Stock Titan (Sent: 0.03)

